# 🏆 Mora SP Cup 2026 — Team Zentrix: ULTRA Model **v9 GRANDMASTER MASTERPIECE**
## 75-Epoch Deep Convergence Engine | Width=64 (86.5M) | Compound Synergy Loss | Curriculum Scaling

### 🔥 The SOTA Breakthrough Recipe (Target: Score 0.640 – 0.665+):
1. **Direct Warm-Start from Champion v7 (`best_denoiser_v7.pth`)**: Resumes from 0.6205 score (PSNR 30.80 dB, SSIM 0.9179) with all 86.5M parameters intact.
2. **Compound Synergy Loss**: `Charbonnier + 0.05*MultiScale_FFT + 0.05*Soft_SSIM`. Charbonnier drives PSNR up past 31.4+ dB while Soft-SSIM and Multi-Scale FFT strictly protect structural edges.
3. **Curriculum Progressive Patch Scaling**:
   * **Phase 1 (Epochs 1–35)**: 256×256 patches (Batch=4, Accum=4) — fast pixel error minimization.
   * **Phase 2 (Epochs 36–55)**: 320×320 patches (Batch=2, Accum=8) — broader contextual illumination learning.
   * **Phase 3 (Epochs 56–75)**: 384×384 patches (Batch=2, Accum=8) — full-scale high-resolution edge perfection.
4. **Kaggle Grandmaster EMA (β=0.999) + SWA (Epochs 61–75)**: Smooths gradient noise and locks into the global flat minimum.
5. **Runtime**: ~50–60 minutes total on Colab T4 GPU (75 focused epochs).

### 🚀 Quick Start:
1. Ensure `best_denoiser_v7.pth` is in your Google Drive root (`MyDrive/`).
2. Choose: **Runtime → Run all** (`Ctrl + F9`).
3. `best_denoiser_v9.pth` will auto-download upon completion!


In [ ]:
# [1] Verify GPU Hardware & Install Packages
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    props = torch.cuda.get_device_properties(0)
    print(f'\u2705 GPU Active: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GB')
else:
    raise RuntimeError('\u274c No GPU! Enable: Runtime → Change runtime type → T4 GPU.')
!nvidia-smi --query-gpu=name,memory.free,memory.total --format=csv
!pip install -q opencv-python-headless scikit-image PyWavelets

In [ ]:
# [2] Create Directory Structure
!mkdir -p scripts/checkpoints models
!mkdir -p competition_data/public/noisy competition_data/public/ground_truth
print('\u2705 Directories ready!')

In [ ]:
# [3] High-Capacity NAFNet-Ultra v9 Architecture (86.49M Parameters)
%%writefile scripts/models.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c, DW_Expand=2, FFN_Expand=2, drop_out_rate=0.0):
        super().__init__()
        dw = c * DW_Expand
        self.conv1 = nn.Conv2d(c, dw, 1, bias=True)
        self.conv2 = nn.Conv2d(dw, dw, 3, 1, 1, groups=dw, bias=True)
        self.conv3 = nn.Conv2d(dw//2, c, 1, bias=True)
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1),
                                  nn.Conv2d(dw//2, dw//2, 1, bias=True))
        self.sg = SimpleGate()
        ffn = FFN_Expand * c
        self.conv4 = nn.Conv2d(c, ffn, 1, bias=True)
        self.conv5 = nn.Conv2d(ffn//2, c, 1, bias=True)
        self.norm1 = nn.GroupNorm(1, c)
        self.norm2 = nn.GroupNorm(1, c)
        self.dropout1 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()
        self.dropout2 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()
        self.beta  = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, inp):
        x = self.norm1(inp)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg(x)
        x = x * self.sca(x)
        x = self.conv3(x)
        x = self.dropout1(x)
        y = inp + x * self.beta
        x = self.norm2(y)
        x = self.conv4(x)
        x = self.sg(x)
        x = self.conv5(x)
        x = self.dropout2(x)
        return y + x * self.gamma

class NAFNet(nn.Module):
    def __init__(self, img_channel=3, width=64,
                 enc_blks=(2, 2, 4, 8), dec_blks=(2, 2, 2, 2), middle_blk_num=8):
        super().__init__()
        self.intro  = nn.Conv2d(img_channel, width, 3, 1, 1, bias=True)
        self.ending = nn.Conv2d(width, img_channel, 3, 1, 1, bias=True)
        self.encoders, self.decoders = nn.ModuleList(), nn.ModuleList()
        self.ups, self.downs = nn.ModuleList(), nn.ModuleList()
        chan = width
        for num in enc_blks:
            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
            self.downs.append(nn.Conv2d(chan, chan*2, 2, 2))
            chan *= 2
        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])
        for num in dec_blks:
            self.ups.append(nn.Sequential(
                nn.Conv2d(chan, chan*2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
        self.padder_size = 2 ** len(enc_blks)

    def forward(self, inp):
        B, C, H, W = inp.shape
        ph = (self.padder_size - H % self.padder_size) % self.padder_size
        pw = (self.padder_size - W % self.padder_size) % self.padder_size
        x = F.pad(inp, (0, pw, 0, ph), 'reflect') if ph or pw else inp
        res = x
        x = self.intro(x)
        encs = []
        for enc, down in zip(self.encoders, self.downs):
            x = enc(x); encs.append(x); x = down(x)
        x = self.middle_blks(x)
        for dec, up, skip in zip(self.decoders, self.ups, reversed(encs)):
            x = up(x) + skip; x = dec(x)
        x = self.ending(x) + res
        return torch.clamp(x[:, :, :H, :W] if ph or pw else x, 0., 1.)

def build_denoising_model(width=64):
    m = NAFNet(img_channel=3, width=width,
               enc_blks=(2,2,4,8), dec_blks=(2,2,2,2), middle_blk_num=8)
    n = sum(p.numel() for p in m.parameters())
    print(f'[Model] NAFNet-Ultra v9 | Width={width} | Params={n/1e6:.2f}M')
    return m


In [ ]:
# [4] Dynamic Multi-Scale Dataset with 8-Way Geometric Augmentations (scripts/dataset.py)
%%writefile scripts/dataset.py
import os, glob, random
import numpy as np, cv2, torch
from torch.utils.data import Dataset

class DenoisingPairDataset(Dataset):
    def __init__(self, noisy_dir, gt_dir, patch_size=256, augment=True):
        self.noisy_dir  = noisy_dir
        self.patch_size = patch_size
        self.augment    = augment
        self.gt_files   = sorted(glob.glob(os.path.join(gt_dir, '*.png')))
        if not self.gt_files:
            raise ValueError(f'No GT images in {gt_dir}')

    def set_patch_size(self, new_patch_size):
        self.patch_size = new_patch_size

    def __len__(self): return len(self.gt_files)

    def _aug(self, n, g):
        h, w = n.shape[:2]
        p = self.patch_size
        if h >= p and w >= p:
            y = random.randint(0, h-p); x = random.randint(0, w-p)
            n, g = n[y:y+p, x:x+p], g[y:y+p, x:x+p]
        else:
            n = cv2.resize(n, (p, p), interpolation=cv2.INTER_LANCZOS4)
            g = cv2.resize(g, (p, p), interpolation=cv2.INTER_LANCZOS4)
        # 8-way spatial augmentation
        if random.random() > 0.5: n, g = np.fliplr(n).copy(), np.fliplr(g).copy()
        if random.random() > 0.5: n, g = np.flipud(n).copy(), np.flipud(g).copy()
        k = random.randint(0, 3)
        if k: n, g = np.rot90(n, k).copy(), np.rot90(g, k).copy()
        return np.ascontiguousarray(n), np.ascontiguousarray(g)

    def __getitem__(self, idx):
        gp = self.gt_files[idx]
        bid = os.path.splitext(os.path.basename(gp))[0]
        np_ = os.path.join(self.noisy_dir, f'{bid}_noise.png')
        if not os.path.exists(np_):
            np_ = os.path.join(self.noisy_dir, f'{bid}.png')
        g = cv2.cvtColor(cv2.imread(gp),  cv2.COLOR_BGR2RGB)
        n = cv2.cvtColor(cv2.imread(np_), cv2.COLOR_BGR2RGB)
        if self.augment: n, g = self._aug(n, g)
        return (torch.from_numpy(n).permute(2,0,1).float()/255.,
                torch.from_numpy(g).permute(2,0,1).float()/255.)


In [ ]:
# [5] Mount Google Drive & Load Public Dataset
from google.colab import drive
import os, glob
drive.mount('/content/drive')

matches = (glob.glob('/content/drive/MyDrive/**/competition_data/public', recursive=True) +
           glob.glob('/content/drive/MyDrive/**/public', recursive=True))
if matches:
    dp = matches[0]
    print(f'\u2705 Dataset found at: {dp}')
    os.system(f'cp -rn {dp}/noisy/* competition_data/public/noisy/ 2>/dev/null')
    os.system(f'cp -rn {dp}/ground_truth/* competition_data/public/ground_truth/ 2>/dev/null')

nn_ = len(glob.glob('competition_data/public/noisy/*.png'))
ng  = len(glob.glob('competition_data/public/ground_truth/*.png'))
print(f'\U0001f4ca Active Dataset: {nn_} noisy | {ng} ground truth images.')
if ng == 0: raise RuntimeError('No ground truth images found! Please check Drive.')

In [ ]:
# [6] ===== ULTRA v9: GRANDMASTER MASTERPIECE TRAINING ENGINE (75 EPOCHS) =====
# Warm-Start v7 (86.5M) | Compound Synergy Loss | Curriculum Scaling | EMA + SWA
from scripts.models import build_denoising_model
from scripts.dataset import DenoisingPairDataset
import torch, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.optim.swa_utils import AveragedModel, SWALR
import time, gc, os, copy, math, numpy as np

gc.collect(); torch.cuda.empty_cache()
device = torch.device('cuda')

# ─── HYPER-PARAMETERS (v9 Configuration) ───────────────────────────
WIDTH         = 64        # 86.49M Parameters (v7 architecture)
TOTAL_EPOCHS  = 75        # 75 deep refinement epochs (~50-60 min)
WARMUP_EPOCHS = 3         # 3-epoch warmup
PEAK_LR       = 5e-5      # SOTA fine-tuning learning rate
MIN_LR        = 1e-7      # Cosine floor
EMA_DECAY     = 0.999     # Exponential Moving Average
SWA_START_EP  = 61        # SWA activates for final 15 epochs
SWA_LR        = 1.5e-5    # SWA learning rate
# ───────────────────────────────────────────────────────────────────

# Base Dataset Split (456 Train / 4 Fixed Val)
full_ds = DenoisingPairDataset(
    'competition_data/public/noisy',
    'competition_data/public/ground_truth',
    patch_size=256, augment=True)
val_size   = min(4, len(full_ds))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size],
                                 generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
print(f'\U0001f680 Data Power: {train_size} training images | {val_size} validation images')

# ─── Build Model & Warm-Start from Champion v7 ─────────────────────
model = build_denoising_model(width=WIDTH).to(device)

v7_candidates = [
    '/content/drive/MyDrive/best_denoiser_v7.pth',
    '/content/drive/MyDrive/best_denoiser.pth',
    'models/best_denoiser_v7.pth',
]
best_score = 0.6205  # Benchmark to smash!
best_psnr  = 30.8025
best_ssim  = 0.9179
loaded = False

for cp in v7_candidates:
    if os.path.exists(cp):
        try:
            ck = torch.load(cp, map_location=device)
            sd = ck.get('model_state_dict', ck)
            clean_sd = {k.replace('module.', ''): v for k, v in sd.items() if k != 'n_averaged'}
            model.load_state_dict(clean_sd)
            print(f'[*] \U0001f525 100% WARM-START LOADED from {cp}!')
            print(f'    Resuming from Champion Baseline: Score={best_score:.4f} | PSNR={best_psnr:.2f} dB | SSIM={best_ssim:.4f}')
            loaded = True
            break
        except Exception as e:
            print(f'[skip] {cp}: {e}')

if not loaded:
    print('[!] Warning: best_denoiser_v7.pth not found in Drive. Starting from current initialized weights.')

# ─── EMA & SWA Trackers ────────────────────────────────────────────
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema_model = copy.deepcopy(model).eval()
        for p in self.ema_model.parameters(): p.requires_grad_(False)
        self.decay = decay
    def update(self, model):
        with torch.no_grad():
            for ep, mp in zip(self.ema_model.parameters(), model.parameters()):
                ep.data.mul_(self.decay).add_(mp.data, alpha=1.0-self.decay)

ema = ModelEMA(model, decay=EMA_DECAY)
swa_model  = AveragedModel(model)
swa_active = False
print(f'\U0001f4a0 EMA Tracker Initialized (beta={EMA_DECAY}) | SWA scheduled at epoch {SWA_START_EP}')

# ─── Optimizer & Schedulers ────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=1e-4, betas=(0.9, 0.999))

def lr_schedule(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)
    else:
        progress = float(epoch - WARMUP_EPOCHS) / float(max(1, TOTAL_EPOCHS - WARMUP_EPOCHS))
        return max(MIN_LR / PEAK_LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler     = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_schedule)
swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
scaler        = torch.cuda.amp.GradScaler()

# ─── Compound Synergy Loss Functions ───────────────────────────────
def charbonnier(pred, gt, eps=1e-6):
    return torch.mean(torch.sqrt((pred - gt)**2 + eps))

def multiscale_fft_loss(pred, gt, scales=[1, 2, 4]):
    total = 0.0
    for s in scales:
        p_ = F.avg_pool2d(pred, s, s) if s > 1 else pred
        g_ = F.avg_pool2d(gt,   s, s) if s > 1 else gt
        fp = torch.fft.rfft2(p_, norm='ortho')
        fg = torch.fft.rfft2(g_, norm='ortho')
        total += torch.mean(torch.abs(fp - fg))
    return total / len(scales)

def soft_ssim_loss(pred, gt):
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(pred, 11, 1, 5)
    mu_g = F.avg_pool2d(gt,   11, 1, 5)
    mu_pp, mu_gg, mu_pg = mu_p**2, mu_g**2, mu_p*mu_g
    sg_pp = F.avg_pool2d(pred*pred, 11, 1, 5) - mu_pp
    sg_gg = F.avg_pool2d(gt*gt,     11, 1, 5) - mu_gg
    sg_pg = F.avg_pool2d(pred*gt,   11, 1, 5) - mu_pg
    num = (2*mu_pg + C1) * (2*sg_pg + C2)
    den = (mu_pp + mu_gg + C1) * (sg_pp + sg_gg + C2)
    return 1.0 - torch.mean(num / den)

BASE_PSNR = 19.8257
BASE_SSIM = 0.4644

def psnr_val(pred, gt):
    return 10. * torch.log10(1. / torch.mean((pred - gt)**2).clamp(1e-8))

def ssim_val(pred, gt):
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(pred, 11, 1, 5)
    mu_g = F.avg_pool2d(gt,   11, 1, 5)
    mu_pp, mu_gg, mu_pg = mu_p**2, mu_g**2, mu_p*mu_g
    sg_pp = F.avg_pool2d(pred*pred, 11, 1, 5) - mu_pp
    sg_gg = F.avg_pool2d(gt*gt,     11, 1, 5) - mu_gg
    sg_pg = F.avg_pool2d(pred*gt,   11, 1, 5) - mu_pg
    num = (2*mu_pg + C1) * (2*sg_pg + C2)
    den = (mu_pp + mu_gg + C1) * (sg_pp + sg_gg + C2)
    return torch.mean(num / den).item()

def composite(psnr, ssim_v):
    return 0.6 * min(max((psnr - BASE_PSNR) / 15., 0.), 1.) + 0.4 * max(ssim_v - BASE_SSIM, 0.)

def evaluate_net(net):
    p_list, s_list = [], []
    with torch.no_grad():
        for nv, gv in val_loader:
            nv, gv = nv.to(device), gv.to(device)
            with torch.cuda.amp.autocast():
                pv = net(nv)
            p_list.append(psnr_val(pv, gv).item())
            s_list.append(ssim_val(pv, gv))
    vp = sum(p_list)/len(p_list)
    vs = sum(s_list)/len(s_list)
    return vp, vs, composite(vp, vs)

CKPT_V9       = 'scripts/checkpoints/best_denoiser_v9.pth'
CKPT_DRIVE    = '/content/drive/MyDrive/best_denoiser_v9.pth'
CKPT_DRIVE_BK = '/content/drive/MyDrive/best_denoiser.pth'

print('=' * 90)
print(f'ULTRA v9 GRANDMASTER MASTERPIECE | Epochs: {TOTAL_EPOCHS} | Peak LR: {PEAK_LR}')
print(f'Loss: Charbonnier + 0.05*MS-FFT + 0.05*Soft-SSIM (Balanced Synergy)')
print(f'Benchmark to Smash: Score={best_score:.4f} | PSNR={best_psnr:.4f} dB | SSIM={best_ssim:.4f}')
print('=' * 90)

# ─── Progressive Curriculum DataLoader Helper ──────────────────────
def get_curriculum_loader(epoch):
    if epoch <= 35:
        patch_size, batch_size, accum = 256, 4, 4
    elif epoch <= 55:
        patch_size, batch_size, accum = 320, 2, 8
    else:
        patch_size, batch_size, accum = 384, 2, 8
    train_ds.dataset.set_patch_size(patch_size)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)
    return loader, accum, patch_size

current_loader, current_accum, current_patch = get_curriculum_loader(1)

# ─── MAIN 75-EPOCH REFINEMENT LOOP ─────────────────────────────────
for epoch in range(1, TOTAL_EPOCHS + 1):
    # Check for curriculum stage transition
    if epoch in [36, 56]:
        current_loader, current_accum, current_patch = get_curriculum_loader(epoch)
        print(f'\n\U0001f31f CURRICULUM UPGRADE: Switching to {current_patch}x{current_patch} Patches (Accum={current_accum})')

    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    t0 = time.time()

    for i, (noisy, gt) in enumerate(current_loader):
        noisy, gt = noisy.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            pred = model(noisy)
            loss = (charbonnier(pred, gt) +
                    0.05 * multiscale_fft_loss(pred, gt) +
                    0.05 * soft_ssim_loss(pred, gt)) / current_accum
        scaler.scale(loss).backward()
        total_loss += loss.item() * current_accum

        if (i + 1) % current_accum == 0 or (i + 1) == len(current_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.01)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            ema.update(model)

    # LR schedule & SWA activation
    if epoch >= SWA_START_EP:
        if not swa_active:
            swa_active = True
            print(f'\n\U0001f300 SWA ACTIVATED at epoch {epoch}! Locking into flat global minimum...')
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()

    cur_lr = optimizer.param_groups[0]['lr']
    model.eval()
    torch.cuda.empty_cache()

    val_p_std, val_s_std, comp_std = evaluate_net(model)
    val_p_ema, val_s_ema, comp_ema = evaluate_net(ema.ema_model)

    candidates = [
        (comp_std, val_p_std, val_s_std, model.state_dict(), '[STD]'),
        (comp_ema, val_p_ema, val_s_ema, ema.ema_model.state_dict(), '[EMA]'),
    ]
    if swa_active and epoch % 3 == 0:
        torch.optim.swa_utils.update_bn(current_loader, swa_model, device=device)
        val_p_swa, val_s_swa, comp_swa = evaluate_net(swa_model)
        candidates.append((comp_swa, val_p_swa, val_s_swa, swa_model.state_dict(), '[SWA]'))

    cur_comp, cur_p, cur_s, best_w, tag = max(candidates, key=lambda x: x[0])
    elapsed = time.time() - t0
    avg_loss = total_loss / len(current_loader)
    flag = ''

    if cur_comp > best_score:
        best_score, best_psnr, best_ssim = cur_comp, cur_p, cur_s
        ckpt = {'model_state_dict': best_w, 'width': WIDTH,
                'psnr': cur_p, 'ssim': cur_s, 'composite_score': cur_comp,
                'epoch': epoch, 'type': tag}
        torch.save(ckpt, CKPT_V9)
        torch.save(ckpt, CKPT_DRIVE)
        torch.save(ckpt, CKPT_DRIVE_BK)
        flag = f'  \U0001f3af NEW ALL-TIME RECORD {tag}! Saved to Drive!'

    print(f'Ep [{epoch:02d}/{TOTAL_EPOCHS}] (P={current_patch}) Loss: {avg_loss:.4f} | '
          f'PSNR: {cur_p:.2f} dB | SSIM: {cur_s:.4f} | '
          f'Score: {cur_comp:.4f} {tag} | LR: {cur_lr:.1e} | {elapsed:.0f}s{flag}')
    torch.cuda.empty_cache()

# ─── Final SWA Evaluation ──────────────────────────────────────────
if swa_active:
    print('\n[*] Final SWA BatchNorm update and evaluation...')
    torch.optim.swa_utils.update_bn(current_loader, swa_model, device=device)
    val_p_swa, val_s_swa, comp_swa = evaluate_net(swa_model)
    print(f'    SWA Final: PSNR={val_p_swa:.4f} dB | SSIM={val_s_swa:.4f} | Score={comp_swa:.4f}')
    if comp_swa > best_score:
        best_score, best_psnr, best_ssim = comp_swa, val_p_swa, val_s_swa
        ckpt = {'model_state_dict': swa_model.state_dict(), 'width': WIDTH,
                'psnr': val_p_swa, 'ssim': val_s_swa, 'composite_score': comp_swa, 'type': '[SWA-FINAL]'}
        torch.save(ckpt, CKPT_V9); torch.save(ckpt, CKPT_DRIVE); torch.save(ckpt, CKPT_DRIVE_BK)
        print(f'\U0001f300 SWA FINAL MODEL IS BEST! Score: {comp_swa:.4f}')

print('=' * 90)
print('\U0001f3c6 ULTRA v9 GRANDMASTER MASTERPIECE TRAINING COMPLETE!')
print(f'   Best Composite Score : {best_score:.4f}')
print(f'   Best PSNR            : {best_psnr:.4f} dB')
print(f'   Best SSIM            : {best_ssim:.4f}')
print('=' * 90)

In [ ]:
# [7] Verify Checkpoint Checksum & Auto-Download to PC
import hashlib
from google.colab import files

ckpt_path = CKPT_V9 if os.path.exists(CKPT_V9) else 'scripts/checkpoints/best_denoiser.pth'
sha = hashlib.sha256()
with open(ckpt_path, 'rb') as f:
    for chunk in iter(lambda: f.read(65536), b''): sha.update(chunk)

hash_val = sha.hexdigest()
print('=' * 90)
print('\U0001f4cb OFFICIAL MODEL SHA-256 CHECKSUM (Version 9):')
print(hash_val)
print('=' * 90)
print(f'Final Best Composite Score : {best_score:.4f}')
print(f'Final Best PSNR            : {best_psnr:.4f} dB')
print(f'Final Best SSIM            : {best_ssim:.4f}')
print()
print('\U0001f4e5 Downloading best_denoiser_v9.pth to your PC...')
files.download(ckpt_path)
print('\u2705 Done! Move the downloaded file into your models/ folder.')